# 04 - Evaluate Models

This notebook loads the processed dataset and saved models, evaluates all three models, saves metrics to JSON, and saves required charts to `reports/figures/`.

In [1]:
from pathlib import Path
import json
import sys
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
if cwd.name == "notebooks":
    BASE_DIR = cwd.parent
elif (cwd / "model building").exists():
    BASE_DIR = cwd / "model building"
else:
    BASE_DIR = cwd

sys.path.append(str(BASE_DIR / "src"))
DATA_PATH = BASE_DIR / "data" / "processed" / "cleansight_processed_dataset.csv"
MODEL_DIR = BASE_DIR / "models"
FIGURES_DIR = BASE_DIR / "reports" / "figures"
METRICS_PATH = BASE_DIR / "reports" / "metrics" / "model_metrics.json"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

decision_tree_model = joblib.load(MODEL_DIR / "decision_tree_cleanliness.pkl")
isolation_forest_model = joblib.load(MODEL_DIR / "isolation_forest_anomaly.pkl")
linear_regression_model = joblib.load(MODEL_DIR / "linear_regression_dust_forecast.pkl")

with open(MODEL_DIR / "model_features.json", "r", encoding="utf-8") as file:
    feature_map = json.load(file)

print("Rows:", len(df))
feature_map

Rows: 1171


{'decision_tree_cleanliness': ['dust',
  'air_quality',
  'temperature',
  'humidity',
  'hour_of_day',
  'dust_rolling_mean_3',
  'air_quality_rolling_mean_3'],
 'isolation_forest_anomaly': ['dust',
  'air_quality',
  'temperature',
  'humidity'],
 'linear_regression_dust_forecast': ['dust_lag_1',
  'air_quality_lag_1',
  'temperature_lag_1',
  'humidity_lag_1',
  'hour_of_day']}

## Save basic sensor trend charts

These charts support the dashboard and presentation by showing how dust and air quality change over time.

In [3]:
from evaluate import save_time_series_plots

save_time_series_plots(df, figures_dir=str(FIGURES_DIR))
print("Saved dust_over_time.png and air_quality_over_time.png")

Saved dust_over_time.png and air_quality_over_time.png


## Evaluate Decision Tree

Metrics: accuracy, classification report, confusion matrix, and feature importance chart.

In [4]:
from evaluate import evaluate_decision_tree

cls_features = feature_map["decision_tree_cleanliness"]
cls_df = df.dropna(subset=cls_features + ["cleanliness_label"]).copy()
X_cls = cls_df[cls_features]
y_cls = cls_df["cleanliness_label"]
stratify = y_cls if y_cls.value_counts().min() >= 2 and y_cls.nunique() > 1 else None

_, X_cls_test, _, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=stratify
)

decision_tree_metrics = evaluate_decision_tree(
    decision_tree_model, X_cls_test, y_cls_test, figures_dir=str(FIGURES_DIR)
)
decision_tree_metrics

{'accuracy': 0.9957446808510638,
 'classification_report': {'clean': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 69.0},
  'dirty': {'precision': 1.0,
   'recall': 0.9879518072289156,
   'f1-score': 0.9939393939393939,
   'support': 83.0},
  'needs_attention': {'precision': 0.9880952380952381,
   'recall': 1.0,
   'f1-score': 0.9940119760479041,
   'support': 83.0},
  'accuracy': 0.9957446808510638,
  'macro avg': {'precision': 0.996031746031746,
   'recall': 0.9959839357429718,
   'f1-score': 0.995983789995766,
   'support': 235.0},
  'weighted avg': {'precision': 0.9957953394123606,
   'recall': 0.9957446808510638,
   'f1-score': 0.9957445264210457,
   'support': 235.0}}}

## Evaluate Isolation Forest

Metrics: normal reading count, anomaly reading count, anomaly percentage, and anomalies-over-time chart.

In [5]:
from evaluate import evaluate_isolation_forest

anomaly_features = feature_map["isolation_forest_anomaly"]
anomaly_df = df.dropna(subset=anomaly_features + ["timestamp"]).copy()
isolation_forest_metrics = evaluate_isolation_forest(
    isolation_forest_model,
    anomaly_df,
    feature_columns=anomaly_features,
    figures_dir=str(FIGURES_DIR),
)
isolation_forest_metrics

{'normal_readings': 1077,
 'anomaly_readings': 94,
 'anomaly_percentage': 8.02732707087959}

## Evaluate Linear Regression

Metrics: MAE, RMSE, R2, and actual-vs-predicted next dust chart.

In [6]:
from evaluate import evaluate_linear_regression

reg_features = feature_map["linear_regression_dust_forecast"]
reg_df = df.dropna(subset=reg_features + ["next_dust"]).copy()
X_reg = reg_df[reg_features]
y_reg = reg_df["next_dust"]

_, X_reg_test, _, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

linear_regression_metrics = evaluate_linear_regression(
    linear_regression_model, X_reg_test, y_reg_test, figures_dir=str(FIGURES_DIR)
)
linear_regression_metrics

{'mae': 9.243714728059498,
 'rmse': 11.692901760773434,
 'r2': 0.9455156007146099}

## Save all metrics

The final metrics JSON can be used in the report and presentation.

In [7]:
from evaluate import save_metrics

metrics = {
    "decision_tree_cleanliness": decision_tree_metrics,
    "isolation_forest_anomaly": isolation_forest_metrics,
    "linear_regression_dust_forecast": linear_regression_metrics,
}

save_metrics(metrics, output_path=str(METRICS_PATH))
print("Saved metrics to:", METRICS_PATH)
print("Saved figures to:", FIGURES_DIR)

Saved metrics to: /Users/vinushan/Documents/VAUED-Project/model building/reports/metrics/model_metrics.json
Saved figures to: /Users/vinushan/Documents/VAUED-Project/model building/reports/figures
